In [2]:
from torch_geometric.utils import to_undirected
import pandas as pd
import torch
from torch_geometric.nn import Node2Vec

import utils
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
df = pd.read_csv('../data/edges/clean_triples.csv')

# 1. Объединяем все значения из обеих колонок
all_entities = pd.concat([df['id_entity_1'], df['id_entity_2']])

# 2. Создаем общий factorize
all_codes, all_uniques = pd.factorize(all_entities)

# 3. Создаем mapping: оригинальный ID → код
id_to_code = dict(zip(all_uniques, range(len(all_uniques))))

# 4. Применяем к обеим колонкам
df['id_entity_1'] = df['id_entity_1'].map(id_to_code)
df['id_entity_2'] = df['id_entity_2'].map(id_to_code)

edge_index = torch.tensor([df['id_entity_1'], df['id_entity_2']])
edge_index = to_undirected(edge_index)

In [4]:
model = Node2Vec(
    edge_index,              # Tensor[2, E] — рёбра графа
    embedding_dim=128,       # d — размерность эмбеддинга
    walk_length=50,          # L — длина случайного блуждания
    context_size=10,         # C — размер контекстного окна
    walks_per_node=10,       # R — число блужданий на узел
    p=1.0, q=1.0,            # параметры возврата/поиска
    num_negative_samples=1,  # k — отрицательные сэмплы
    sparse=True              # разреженные градиенты
)

model.to(device)

Node2Vec(1073652, 128)

In [7]:
loader = model.loader(batch_size=256, shuffle=True, num_workers=4)
optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)

def train():
    model.train()
    for pos_rw, neg_rw in loader:
        pos_rw = pos_rw.to(device)
        neg_rw = neg_rw.to(device)
        optimizer.zero_grad()
        loss = model.loss(pos_rw, neg_rw)
        loss.backward()
        optimizer.step()


In [8]:
for _ in tqdm(range(10)):
    train()

  0%|          | 0/10 [00:32<?, ?it/s]


KeyboardInterrupt: 